# NHANES diabetes project — data pipeline (v7)

**How to run:** Runtime → *Restart session and run all*. Never run cells out of order.

**If any cell fails: don't fix it and don't ask an LLM to rewrite it.** Run the last cell (9), then send Sampreeth the cell number, the full red error, and cell 9's output.

## One naming convention, four layers

| Layer | Names | Where | File |
|---|---|---|---|
| 1. Raw NHANES | `SDMVPSU`, `SDMVSTRA`, `WT…`, `LBXGH`, `DIQ010` … | cells 1–2 only | `nhanes_raw_merged.parquet` (audit / crosswalk) |
| 2. Standardised | `stratum`, `psu`, `w_int`, `w_lab`, `w_fast`, `w`, `age`, `bmi`, `hba1c`, `fpg`, `sbp` … | cell 3 | `nhanes_all.parquet` |
| 3. Analysis variables | `adult`, `y_diag`, `y_bio`, `undx`, `screened`, subgroups | cell 4 | `nhanes_all.parquet` |
| 4. Results | prevalence, CIs, reference checks, counts | cells 5–7 | `step5_…`, `step6_…`, `step7_…` CSVs |

Raw names never appear after cell 4. Cell 8 stops the run if any raw name, renamed column or unexpected value reaches the analysis dataset.

## Which weight goes with what

| Analysis | Label | Weight (one cycle) | Weight (pooled train) |
|---|---|---|---|
| Main biomarker | `y_bio`, `undx` | `w_lab` | `w` |
| Sensitivity | `y_bio_any` | `w_lab` | `w` |
| Fasting glucose sensitivity | `y_bio_fpg`, `undx_fpg` | `w_fast` | `w_fast_pooled` |
| Screening (questionnaire) | `screened` | `w_int` | `w_int_pooled` |

Filter with `adult` and `lab_sample` as domains — **never delete rows before computing estimates.** `hba1c` and `fpg` define the labels, so they must **never** be model inputs.


## 0. Setup

In [1]:
# NHANES diabetes project — data pipeline (v7)
# Naming convention (one direction only):
#   Layer 1 RAW NHANES        SDMVPSU, SDMVSTRA, WT..., LBXGH, DIQ010 ...   (cells 1-2 only)
#   Layer 2 STANDARDISED      psu, stratum, w_int, w_lab, w_fast, age, bmi, hba1c ...  (cell 3)
#   Layer 3 ANALYSIS VARS     adult, y_diag, y_bio, undx, screened, subgroups          (cell 4)
#   Layer 4 RESULTS           prevalence, CIs, counts, comparisons                     (cells 5-7)
# Raw names never appear after cell 4. Outputs:
#   nhanes_out/nhanes_raw_merged.parquet  (layer 1, for audit / crosswalk only)
#   nhanes_out/nhanes_all.parquet         (layers 2+3, what everyone analyses)
import os
import re

import numpy as np
import pandas as pd
import requests
from scipy import stats

# Optional (recommended): save everything to Google Drive so it survives resets
# from google.colab import drive; drive.mount('/content/drive')
# os.makedirs('/content/drive/MyDrive/nhanes_project', exist_ok=True)
# os.chdir('/content/drive/MyDrive/nhanes_project')

RAW_DIR, OUT_DIR = "nhanes_raw", "nhanes_out"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# cycle -> (URL year, file prefix, file suffix, legacy folder, period)
CYCLES = {
    "2011-2012": (2011, "",   "_G", "2011-2012", "train"),
    "2013-2014": (2013, "",   "_H", "2013-2014", "train"),
    "2015-2016": (2015, "",   "_I", "2015-2016", "train"),
    "2017-2020": (2017, "P_", "",   "2017-2018", "test_P"),  # pre-pandemic file; never split it
    "2021-2023": (2021, "",   "_L", "2021-2022", "test_L"),
}
TRAIN_CYCLES = [c for c, v in CYCLES.items() if v[4] == "train"]

# file -> columns to keep (None = keep every column). Any column starting with "WT" is always kept.
FILES = {
    "DEMO":   ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "INDFMPIR",
               "RIDEXPRG", "SDMVSTRA", "SDMVPSU"],
    "DIQ":    ["SEQN", "DIQ010", "DIQ159", "DIQ160", "DIQ180", "DIQ050", "DIQ070"],
    "HIQ":    ["SEQN", "HIQ011"],
    "HUQ":    ["SEQN", "HUQ030"],
    "INQ":    None,
    "BMX":    ["SEQN", "BMXBMI", "BMXWAIST"],
    "GHB":    ["SEQN", "LBXGH"],
    "GLU":    ["SEQN", "LBXGLU"],
    "FASTQX": ["SEQN", "PHAFSTHR"],
    "BP":     None,   # BPX (2011-2016, auscultatory) or BPXO (2017+, oscillometric)
}
BP_FILE = {"2011-2012": "BPX", "2013-2014": "BPX", "2015-2016": "BPX",
           "2017-2020": "BPXO", "2021-2023": "BPXO"}

HEADERS = {"User-Agent": "Mozilla/5.0 (research download script)"}

# Required components for this analysis. If an official cycle file is unavailable,
# stop and document the exception rather than silently creating an incomplete merge.
REQUIRED_BASES = list(FILES.keys())

## 1. LAYER 1 (raw): download

In [2]:
def file_name(base, cycle):
    _, prefix, suffix, _, _ = CYCLES[cycle]
    if base == "BP":
        base = BP_FILE[cycle]
    return f"{prefix}{base}{suffix}"


def candidate_urls(name, cycle):
    year, _, _, legacy, _ = CYCLES[cycle]
    return [
        f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}.xpt",
        f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}.XPT",
        f"https://wwwn.cdc.gov/Nchs/Nhanes/{legacy}/{name}.XPT",
    ]


def is_xpt_file(path):
    try:
        with open(path, "rb") as f:
            return f.read(13) == b"HEADER RECORD"
    except OSError:
        return False


def download(base, cycle):
    """Download one NHANES XPT file; return a verified local path or raise."""
    name = file_name(base, cycle)
    path = os.path.join(RAW_DIR, f"{name}.xpt")
    if os.path.exists(path) and os.path.getsize(path) > 0 and is_xpt_file(path):
        return path

    # Remove a stale/invalid cached file before trying again.
    if os.path.exists(path):
        os.remove(path)

    errors = []
    for url in candidate_urls(name, cycle):
        try:
            r = requests.get(url, headers=HEADERS, timeout=60)
            if r.status_code == 200 and r.content[:13] == b"HEADER RECORD":
                with open(path, "wb") as f:
                    f.write(r.content)
                return path
            errors.append(f"{url} -> HTTP {r.status_code}")
        except requests.RequestException as exc:
            errors.append(f"{url} -> {exc}")

    raise RuntimeError(
        f"Required file {name} for {cycle} could not be downloaded. "
        "Check the official NHANES cycle page and update the required-file "
        f"configuration. Attempts: {'; '.join(errors)}"
    )


def read_xpt(path, wanted):
    df = pd.read_sas(path, format="xport", encoding="latin-1")
    df.columns = [c.upper() for c in df.columns]
    num = df.select_dtypes("number").columns
    df[num] = df[num].mask(df[num].abs() < 1e-20, 0.0)
    if wanted is None:
        return df
    keep = [c for c in wanted if c in df.columns]
    keep += [c for c in df.columns if c.startswith("WT")]
    return df[list(dict.fromkeys(keep))]


raw = {}
download_audit = []
for cycle in CYCLES:
    print(f"Downloading {cycle} ...")
    for base, cols in FILES.items():
        path = download(base, cycle)  # raises if a required file is missing
        frame = read_xpt(path, cols)
        if "SEQN" not in frame.columns:
            raise ValueError(f"{file_name(base, cycle)} has no SEQN column")
        if frame["SEQN"].duplicated().any():
            dup_count = int(frame["SEQN"].duplicated().sum())
            raise ValueError(
                f"{file_name(base, cycle)} contains {dup_count} duplicate SEQN values"
            )
        raw[(base, cycle)] = frame
        download_audit.append({
            "cycle": cycle, "component": base, "file": os.path.basename(path),
            "rows": len(frame), "duplicate_seqn": int(frame["SEQN"].duplicated().sum())
        })

download_audit = pd.DataFrame(download_audit)
download_audit.to_csv(os.path.join(OUT_DIR, "step1_download_audit.csv"), index=False)
print(download_audit.to_string(index=False))
print("All configured required files downloaded and passed basic XPT/SEQN checks.")

    cycle component         file  rows  duplicate_seqn
2011-2012      DEMO   DEMO_G.xpt  9756               0
2011-2012       DIQ    DIQ_G.xpt  9364               0
2011-2012       HIQ    HIQ_G.xpt  9756               0
2011-2012       HUQ    HUQ_G.xpt  9756               0
2011-2012       INQ    INQ_G.xpt  9756               0
2011-2012       BMX    BMX_G.xpt  9338               0
2011-2012       GHB    GHB_G.xpt  6549               0
2011-2012       GLU    GLU_G.xpt  3239               0
2011-2012    FASTQX FASTQX_G.xpt  8956               0
2011-2012        BP    BPX_G.xpt  9338               0
2013-2014      DEMO   DEMO_H.xpt 10175               0
2013-2014       DIQ    DIQ_H.xpt  9770               0
2013-2014       HIQ    HIQ_H.xpt 10175               0
2013-2014       HUQ    HUQ_H.xpt 10175               0
2013-2014       INQ    INQ_H.xpt 10175               0
2013-2014       BMX    BMX_H.xpt  9813               0
2013-2014       GHB    GHB_H.xpt  6979               0
2013-2014 

## 2. LAYER 1 (raw): merge and clean special codes

In [3]:
def merge_cycle(cycle):
    if ("DEMO", cycle) not in raw:
        raise RuntimeError(f"DEMO missing for {cycle}")

    df = raw[("DEMO", cycle)].copy()
    merge_audit = []

    for base in FILES:
        if base == "DEMO":
            continue
        if (base, cycle) not in raw:
            raise RuntimeError(f"Required component {base} missing for {cycle}")

        other = raw[(base, cycle)].copy()
        overlap = sorted((set(df.columns) & set(other.columns)) - {"SEQN"})
        for col in overlap:
            # Only discard a duplicate if values agree for every shared SEQN.
            check = df[["SEQN", col]].merge(
                other[["SEQN", col]], on="SEQN", how="inner",
                suffixes=("_left", "_right")
            )
            left, right = check[f"{col}_left"], check[f"{col}_right"]
            equal = left.eq(right) | (left.isna() & right.isna())
            if not bool(equal.all()):
                raise ValueError(
                    f"Conflicting duplicate column {col!r} while merging "
                    f"{base} into {cycle}; inspect source files."
                )
            other = other.drop(columns=[col])
            merge_audit.append({
                "cycle": cycle, "component": base, "duplicate_column": col,
                "action": "dropped after exact shared-SEQN equality check"
            })

        before = len(df)
        df = df.merge(other, on="SEQN", how="left", validate="one_to_one")
        if len(df) != before:
            raise AssertionError(
                f"Row count changed from {before} to {len(df)} during {base}/{cycle} merge"
            )

    df["cycle"] = cycle
    df["period"] = CYCLES[cycle][4]
    df["bp_method"] = "oscillometric" if BP_FILE[cycle] == "BPXO" else "auscultatory"
    return df, merge_audit


merged_cycles = []
merge_audits = []
for cycle in CYCLES:
    cycle_df, audit_rows = merge_cycle(cycle)
    merged_cycles.append(cycle_df)
    merge_audits.extend(audit_rows)

df = pd.concat(merged_cycles, ignore_index=True)
df["SEQN"] = df["SEQN"].astype("int64")
if df.duplicated(["cycle", "SEQN"]).any():
    raise AssertionError("Duplicate participant IDs found within a cycle after merging")

pd.DataFrame(merge_audits).to_csv(
    os.path.join(OUT_DIR, "step2_merge_overlap_audit.csv"), index=False
)
print("Merged rows by cycle:")
print(df.groupby("cycle").size().to_string())
print("Merge validation passed: one-to-one joins and unchanged DEMO row counts.")

# --- which variables exist in which cycle (0 = absent in that cycle)
check_vars = ["DIQ010", "DIQ180", "HIQ011", "HUQ030", "LBXGH", "LBXGLU", "BMXBMI", "INDFMPIR"]
avail = df.groupby("cycle")[check_vars].apply(lambda g: g.notna().mean().round(3))
print("\nShare of non-missing values per cycle:")
print(avail.to_string())

# --- special codes: Refused / Don't know -> missing (cell 3 repeats this safely)
SPECIAL = [7, 9, 77, 99, 777, 999, 7777, 9999]
CODED_VARS = ["DIQ010", "DIQ160", "DIQ180", "DIQ050", "DIQ070", "HIQ011", "HUQ030", "DMDEDUC2"]
for col in CODED_VARS:
    if col in df:
        df[col] = df[col].where(~df[col].isin(SPECIAL))


def save(frame, name):
    try:
        frame.to_parquet(os.path.join(OUT_DIR, f"{name}.parquet"), index=False)
        print(f"Saved {len(frame):,} rows -> {OUT_DIR}/{name}.parquet")
    except Exception as e:   # pyarrow missing
        frame.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=False)
        print(f"Parquet failed ({e}); saved {name}.csv instead")


save(df, "nhanes_raw_merged")

Merged rows by cycle:
cycle
2011-2012     9756
2013-2014    10175
2015-2016     9971
2017-2020    15560
2021-2023    11933
Merge validation passed: one-to-one joins and unchanged DEMO row counts.

Share of non-missing values per cycle:
           DIQ010  DIQ180  HIQ011  HUQ030  LBXGH  LBXGLU  BMXBMI  INDFMPIR
cycle                                                                     
2011-2012   0.960   0.628   1.000     1.0  0.630   0.311   0.882     0.914
2013-2014   0.960   0.636   1.000     1.0  0.653   0.312   0.890     0.923
2015-2016   0.960   0.620   1.000     1.0  0.634   0.298   0.878     0.894
2017-2020   0.963   0.630   1.000     1.0  0.626   0.305   0.844     0.859
2021-2023   0.984   0.696   0.998     1.0  0.563   0.308   0.710     0.829
Parquet failed (Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` f

## 3. LAYER 2: standardised dataset

In [4]:
# Every raw NHANES name is translated here ONCE. After this cell, use only the lowercase names.
# Re-apply special-code cleaning so this cell never depends on what cell 2 looked like.
for col in ["DIQ010", "DIQ160", "DIQ180", "DIQ050", "DIQ070", "HIQ011", "HUQ030", "DMDEDUC2"]:
    if col in df:
        df[col] = df[col].where(~df[col].isin([7, 9, 77, 99, 777, 999, 7777, 9999]))

# --- design: strata/PSU numbers repeat across cycles -> make them unique
for col in ["SDMVSTRA", "SDMVPSU"]:
    if df[col].isna().any():
        raise ValueError(f"{col} has missing values — every participant needs a stratum and PSU")
df["stratum"] = df["cycle"] + "_" + df["SDMVSTRA"].astype("Int64").astype(str)
df["psu"] = df["stratum"] + "_" + df["SDMVPSU"].astype("Int64").astype(str)

# --- weights: the exact column CDC documents for each cycle (stops if any is absent)
WEIGHT_MAP = {   # standard name -> {cycle: raw NHANES column}
    "w_int":  {"2011-2012": "WTINT2YR", "2013-2014": "WTINT2YR", "2015-2016": "WTINT2YR",
               "2017-2020": "WTINTPRP", "2021-2023": "WTINT2YR"},   # interview / questionnaire
    "w_mec":  {"2011-2012": "WTMEC2YR", "2013-2014": "WTMEC2YR", "2015-2016": "WTMEC2YR",
               "2017-2020": "WTMECPRP", "2021-2023": "WTMEC2YR"},   # exam
    "w_lab":  {"2011-2012": "WTMEC2YR", "2013-2014": "WTMEC2YR", "2015-2016": "WTMEC2YR",
               "2017-2020": "WTMECPRP", "2021-2023": "WTPH2YR"},    # HbA1c (blood draw in 2021-23)
    "w_fast": {"2011-2012": "WTSAF2YR", "2013-2014": "WTSAF2YR", "2015-2016": "WTSAF2YR",
               "2017-2020": "WTSAFPRP", "2021-2023": "WTSAF2YR"},   # fasting glucose; 0 outside 8-<24 h
}
for std, per_cycle in WEIGHT_MAP.items():
    out = pd.Series(np.nan, index=df.index)
    for cyc, raw_col in per_cycle.items():
        m = df["cycle"].eq(cyc)
        if raw_col not in df.columns or df.loc[m, raw_col].isna().all():
            raise KeyError(f"{cyc}: expected weight {raw_col} (for {std}) not found — check the codebook")
        out[m] = df.loc[m, raw_col]
    df[std] = out

# --- pooled weights (CDC: three 2-year cycles -> each x 1/3). 2017-2020 and 2021-23 are never pooled.
POOL_FACTOR = {c: (2 / 6 if c in TRAIN_CYCLES else 1.0) for c in CYCLES}
df["pool_factor"] = df["cycle"].map(POOL_FACTOR)
assert (df.loc[df["period"] != "train", "pool_factor"] == 1).all()
df["w"] = df["w_lab"] * df["pool_factor"]               # main pooled weight (HbA1c analyses)
df["w_int_pooled"] = df["w_int"] * df["pool_factor"]
df["w_fast_pooled"] = df["w_fast"] * df["pool_factor"]

# --- standardised measurements / features (Rushmitha: extend FEATURE_MAP here, nowhere else)
FEATURE_MAP = {"age": "RIDAGEYR", "pir": "INDFMPIR", "education": "DMDEDUC2",
               "bmi": "BMXBMI", "waist": "BMXWAIST",
               "hba1c": "LBXGH",      # evaluation only — NEVER a model input (it defines y_bio)
               "fpg": "LBXGLU"}       # evaluation only — values as released by CDC (no adjustment):
# GLU_G/GLU_H describe no regression adjustment; the GLU_I C501->C311 equation applies only to that
# lab's C501 results, which CDC already adjusted in GLU_I. NCHS used released values too.
for std, raw_col in FEATURE_MAP.items():
    df[std] = pd.to_numeric(df[raw_col], errors="coerce") if raw_col in df.columns else np.nan

# blood pressure: mean of available readings. Method changed (bp_method) -> not comparable over time.
sys_cols = [c for c in df.columns if re.fullmatch(r"BPX(O)?SY\d", c)]
dia_cols = [c for c in df.columns if re.fullmatch(r"BPX(O)?DI\d", c)]
df["sbp"] = df[sys_cols].mean(axis=1) if sys_cols else np.nan
df["dbp"] = df[dia_cols].where(df[dia_cols] > 0).mean(axis=1) if dia_cols else np.nan   # 0 = not audible

# --- weight audit: each period should sum to about one US population (train NOT 3x)
audit = (df.assign(pos_lab=df["w_lab"] > 0, pos_fast=df["w_fast"] > 0)
           .groupby("period", sort=False)
           .agg(cycles=("cycle", lambda s: ", ".join(sorted(s.unique()))),
                n=("SEQN", "size"), n_lab_weight_pos=("pos_lab", "sum"),
                n_fast_weight_pos=("pos_fast", "sum"),
                sum_w_millions=("w", lambda s: round(s.sum() / 1e6, 1)),
                sum_w_int_pooled_millions=("w_int_pooled", lambda s: round(s.sum() / 1e6, 1))))
audit.to_csv(os.path.join(OUT_DIR, "step3_weight_audit.csv"))
print("Weight audit (sums in millions of people):")
print(audit.to_string())

Weight audit (sums in millions of people):
                                 cycles      n  n_lab_weight_pos  n_fast_weight_pos  sum_w_millions  sum_w_int_pooled_millions
period                                                                                                                        
train   2011-2012, 2013-2014, 2015-2016  29902             28695               8512           311.4                      311.4
test_P                        2017-2020  15560             14300               4476           322.3                      322.3
test_L                        2021-2023  11933              6750               3361           280.0                      327.0


## 4. LAYER 3: analysis variables

In [5]:
# Built from the standardised layer, plus the categorical questionnaire answers (recoded here once).

# --- analysis population: adults 20+, not pregnant. Use as a DOMAIN filter — never drop rows.
pregnant = df["RIDEXPRG"].eq(1) if "RIDEXPRG" in df else False
df["adult"] = (df["age"] >= 20) & ~pregnant

# --- diagnosis label: DIQ010 1 = yes; 2 = no and 3 = borderline both count as "not diagnosed" (NCHS)
df["y_diag"] = df["DIQ010"].map({1: 1.0, 2: 0.0, 3: 0.0})
has_a1c = df["hba1c"].notna()
a1c_pos = df["hba1c"] >= 6.5
answered = df["y_diag"].notna()

# PRIMARY y_bio: answered diagnosis question AND valid HbA1c, weight w_lab. Diagnosed people without
# HbA1c are not kept, because undiagnosed people without HbA1c can't be classified and must be dropped;
# keeping only one side would overstate diabetes. Same population NCHS uses.
df["lab_sample"] = answered & has_a1c
df["y_bio"] = np.where(df["lab_sample"], ((df["y_diag"] == 1) | a1c_pos).astype(float), np.nan)
df["undx"] = np.where(df["lab_sample"], ((df["y_diag"] == 0) & a1c_pos).astype(float), np.nan)

# SENSITIVITY y_bio_any: diagnosed = positive even without HbA1c. Report next to y_bio, never instead.
df["y_bio_any"] = np.select([df["y_diag"].eq(1), df["y_diag"].eq(0) & has_a1c],
                            [1.0, a1c_pos.astype(float)], default=np.nan)

# SENSITIVITY y_bio_fpg: HbA1c OR fasting glucose, fasting subsample only (w_fast > 0), weight w_fast
ok_f = df["lab_sample"] & df["fpg"].notna() & (df["w_fast"].fillna(0) > 0)
bio_f = a1c_pos | (df["fpg"] >= 126)
df["y_bio_fpg"] = np.where(ok_f, ((df["y_diag"] == 1) | bio_f).astype(float), np.nan)
df["undx_fpg"] = np.where(ok_f, ((df["y_diag"] == 0) & bio_f).astype(float), np.nan)
df["y_diag_fpg"] = np.where(ok_f, df["y_diag"], np.nan)

# --- screened: DIQ180 "blood test for high blood sugar in the past three years". Asked only of people
# without a diagnosis (routed by check item DIQ159). Questionnaire item -> main estimate uses w_int.
routing = df.loc[df["DIQ010"].eq(1)].groupby("cycle")["DIQ180"].apply(lambda s: int(s.notna().sum()))
print("DIQ180 answers from DIAGNOSED people, per cycle (expected 0):", routing.to_dict())
if routing.sum() > 0:
    print("  -> check that cycle's DIQ codebook. 'screened' is still safe: it only uses y_diag == 0.")
df["screened"] = np.where(df["y_diag"].eq(0), df["DIQ180"].map({1: 1.0, 2: 0.0}), np.nan)

# --- access: HUQ030 usual place for care. Codes 1 yes, 2 no place, 3 more than one place (7/9 missing).
bad_codes = sorted(set(df["HUQ030"].dropna().unique()) - {1, 2, 3})
if bad_codes:
    raise ValueError(f"HUQ030 has undocumented codes {bad_codes} — check that cycle's HUQ codebook")
df["usual_source_care"] = df["HUQ030"].map({1: 1.0, 3: 1.0, 2: 0.0})
df["routine_care"] = df["usual_source_care"]          # alias kept for older code
df["insured"] = df["HIQ011"].map({1: 1.0, 2: 0.0})

# --- subgroups (cut points agreed; change only by team agreement)
df["income_band"] = pd.cut(df["pir"], [-np.inf, 1.3, 3.5, np.inf], labels=["<1.3", "1.3-3.5", ">3.5"], right=False)
df["income_band2"] = pd.cut(df["pir"], [-np.inf, 1.3, np.inf], labels=["<1.3", ">=1.3"], right=False)
df["age_band"] = pd.cut(df["age"], [20, 40, 60, np.inf], labels=["20-39", "40-59", "60+"], right=False)
df["age_band2"] = pd.cut(df["age"], [20, 60, np.inf], labels=["20-59", "60+"], right=False)
df["sex"] = df["RIAGENDR"].map({1: "male", 2: "female"})
df["race_eth"] = df["RIDRETH3"].map({1: "Mexican American", 2: "Other Hispanic", 3: "NH White",
                                     4: "NH Black", 6: "NH Asian", 7: "Other/Multi"})
df["race_eth4"] = df["RIDRETH3"].map({1: "Hispanic", 2: "Hispanic", 3: "NH White", 4: "NH Black",
                                      6: "NH Asian/Other", 7: "NH Asian/Other"})

# --- build the analysis dataset: ONLY layer 2 + layer 3 columns
LAYER2 = ["SEQN", "cycle", "period", "stratum", "psu",
          "w_int", "w_mec", "w_lab", "w_fast", "pool_factor", "w", "w_int_pooled", "w_fast_pooled",
          "age", "pir", "education", "bmi", "waist", "sbp", "dbp", "bp_method", "hba1c", "fpg"]
LAYER3 = ["adult", "lab_sample", "y_diag", "y_bio", "y_bio_any", "undx", "y_bio_fpg", "undx_fpg",
          "y_diag_fpg", "screened", "insured", "usual_source_care", "routine_care",
          "income_band", "income_band2", "age_band", "age_band2", "sex", "race_eth", "race_eth4"]
nhanes_all = df[LAYER2 + LAYER3].copy()
save(nhanes_all, "nhanes_all")

# --- how much the y_bio choice matters: missing HbA1c among examined adults, by diagnosis
ex = nhanes_all[(nhanes_all["w_mec"].fillna(0) > 0) & nhanes_all["adult"] & nhanes_all["y_diag"].notna()]
miss = (ex.assign(missing_hba1c=ex["hba1c"].isna()).groupby(["cycle", "y_diag"])["missing_hba1c"]
          .agg(n="size", n_missing="sum").reset_index())
miss["pct_missing"] = (100 * miss["n_missing"] / miss["n"]).round(1)
miss.to_csv(os.path.join(OUT_DIR, "step4_missing_hba1c.csv"), index=False)
print("\nMissing HbA1c among examined adults (y_diag 0 = not diagnosed, 1 = diagnosed):")
print(miss.to_string(index=False))

DIQ180 answers from DIAGNOSED people, per cycle (expected 0): {'2011-2012': 0, '2013-2014': 0, '2015-2016': 0, '2017-2020': 0, '2021-2023': 0}
Parquet failed (Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.); saved nhanes_all.csv instead

Missing HbA1c among examined adults (y_diag 0 = not diagnosed, 1 = diagnosed):
    cycle  y_diag    n  n_missing  pct_missing
2011-2012     0.0 4582        256          5.6
2011-2012     1.0  676         36          5.3
2013-2014     0.0 4815        167          3.5
2013-2014     1.0  705         25          3.5
2015-2016     0.0 4595        242          5.

## 5. LAYER 4 results: weighted prevalence, Taylor SE, t-based CI

In [6]:
# Results use ONLY the analysis dataset (nhanes_all), never raw NHANES names.
def svy_prop(d, y, w, domain=None):
    """Weighted proportion of binary y for ONE cycle; domain rows are zeroed, not dropped."""
    in_dom = (d[w].fillna(0) > 0) & d[y].notna()
    if domain is not None:
        in_dom &= domain
    wt = np.where(in_dom, d[w].fillna(0), 0.0)
    yv = np.where(in_dom, d[y].fillna(0), 0.0)
    W = wt.sum()
    if W == 0:
        return np.nan, np.nan, 0
    p = (wt * yv).sum() / W
    z = wt * (yv - p) / W
    tot = pd.DataFrame({"h": d["stratum"].values, "psu": d["psu"].values, "z": z}) \
            .groupby(["h", "psu"])["z"].sum().reset_index()
    var = 0.0
    for _, g in tot.groupby("h"):
        n = len(g)
        if n > 1:
            var += n / (n - 1) * ((g["z"] - g["z"].mean()) ** 2).sum()
    return p, np.sqrt(var), int(in_dom.sum())


MEASURES = [  # (label, variable, weight, extra domain)
    ("Diagnosed, lab sample",            "y_diag",     "w_lab",  "lab_sample"),
    ("Total, HbA1c (y_bio, PRIMARY)",    "y_bio",      "w_lab",  None),
    ("Total, HbA1c (y_bio_any, sens.)",  "y_bio_any",  "w_lab",  None),
    ("Undiagnosed, HbA1c",               "undx",       "w_lab",  None),
    ("Diagnosed, interview sample",      "y_diag",     "w_int",  None),
    ("Screened in 3y, among undiagnosed (interview weight, MAIN)", "screened", "w_int", None),
    ("Screened in 3y, among undiagnosed (lab sample, for joint analyses)", "screened", "w_lab", "lab_sample"),
    ("NCHS-style: Diagnosed",            "y_diag_fpg", "w_fast", None),
    ("NCHS-style: Total (HbA1c or FPG)", "y_bio_fpg",  "w_fast", None),
    ("NCHS-style: Undiagnosed",          "undx_fpg",   "w_fast", None),
]

rows = []
for cycle, d in nhanes_all.groupby("cycle"):           # full cycle -> all PSUs stay in the design
    dof = d["psu"].nunique() - d["stratum"].nunique()      # FIX 2: design degrees of freedom
    t = stats.t.ppf(0.975, dof)
    for label, y, w, extra in MEASURES:
        dom = d["adult"] & (d[extra] if extra else True)
        p, se, n = svy_prop(d, y, w, dom)
        rows.append({"cycle": cycle, "measure": label, "weight": w, "n": n, "dof": dof,
                     "pct": 100 * p, "se": 100 * se,
                     "ci_low": 100 * (p - t * se), "ci_high": 100 * (p + t * se)})
prev = pd.DataFrame(rows)
prev.to_csv(os.path.join(OUT_DIR, "step5_weighted_prevalence.csv"), index=False)
print("Weighted prevalence, adults 20+ (%):")
print(prev.pivot(index="measure", columns="cycle", values="pct").round(1).to_string())
print("\nDesign degrees of freedom:", prev.groupby("cycle")["dof"].first().to_dict())

Weighted prevalence, adults 20+ (%):
cycle                                                               2011-2012  2013-2014  2015-2016  2017-2020  2021-2023
measure                                                                                                                  
Diagnosed, interview sample                                               9.2       10.0       11.1       11.7       11.2
Diagnosed, lab sample                                                     9.3       10.2       11.4       11.8       11.8
NCHS-style: Diagnosed                                                     9.2       10.0       11.1       11.7       11.3
NCHS-style: Total (HbA1c or FPG)                                         13.2       13.7       16.1       16.2       15.8
NCHS-style: Undiagnosed                                                   4.0        3.7        5.0        4.5        4.5
Screened in 3y, among undiagnosed (interview weight, MAIN)               47.8       52.0       55.5       53.

## 6. LAYER 4 results: reference checks against NCHS

In [7]:
# NCHS Data Brief 516, Aug 2021–Aug 2023, adults 20+, crude %, pregnant excluded,
# fasting subsample weights. https://www.cdc.gov/nchs/products/databriefs/db516.htm
# To validate another cycle, add its crude values from the matching NCHS data brief.
NCHS = {
    ("2021-2023", "NCHS-style: Total (HbA1c or FPG)"): (15.8, 13.6, 18.2),
    ("2021-2023", "NCHS-style: Diagnosed"):            (11.3, 9.3, 13.5),
    ("2021-2023", "NCHS-style: Undiagnosed"):          (4.5, 3.3, 5.9),
}
val = []
for (cycle, measure), (pub, lo, hi) in NCHS.items():
    r = prev[(prev["cycle"] == cycle) & (prev["measure"] == measure)]
    if r.empty:
        continue
    ours = r["pct"].iloc[0]
    val.append({"cycle": cycle, "measure": measure, "ours_pct": round(ours, 1),
                "nchs_pct": pub, "nchs_ci": f"{lo}-{hi}",
                "match": "PASS" if lo <= ours <= hi else "FAIL"})
# --- Check 2: pooled 2013-2016 (a TRAINING period) vs NCHS Data Brief 319
# https://www.cdc.gov/nchs/products/databriefs/db319.htm — adults 20+, crude: total 15.0, diagnosed 10.5,
# undiagnosed 4.5. The brief gives no CI, so PASS = the NCHS value lies inside OUR 95% CI.
# Also tests the pooling rule: two 2-year cycles -> fasting weight x 1/2.
p13 = nhanes_all[nhanes_all["cycle"].isin(["2013-2014", "2015-2016"])].copy()
p13["w_fast_4yr"] = p13["w_fast"] / 2
okp = p13["y_diag"].notna() & p13["hba1c"].notna() & p13["fpg"].notna() & (p13["w_fast_4yr"].fillna(0) > 0)
biop = (p13["hba1c"] >= 6.5) | (p13["fpg"] >= 126)
p13["v_diag"] = np.where(okp, p13["y_diag"], np.nan)
p13["v_total"] = np.where(okp, ((p13["y_diag"] == 1) | biop).astype(float), np.nan)
p13["v_undx"] = np.where(okp, ((p13["y_diag"] == 0) & biop).astype(float), np.nan)
dof13 = p13["psu"].nunique() - p13["stratum"].nunique()
t13 = stats.t.ppf(0.975, dof13)
for measure, var, pub in [("2013-16 pooled: Total", "v_total", 15.0),
                          ("2013-16 pooled: Diagnosed", "v_diag", 10.5),
                          ("2013-16 pooled: Undiagnosed", "v_undx", 4.5)]:
    p, se, n = svy_prop(p13, var, "w_fast_4yr", p13["adult"])
    lo, hi = 100 * (p - t13 * se), 100 * (p + t13 * se)
    val.append({"cycle": "2013-2016", "measure": measure, "ours_pct": round(100 * p, 1),
                "nchs_pct": pub, "nchs_ci": f"ours {lo:.1f}-{hi:.1f}",
                "match": "PASS" if lo <= pub <= hi else "FAIL"})

# --- Check 3: 2017-March 2020. Add a published estimate here once a team member finds one
# with the SAME definition (adults 20+, crude, fasting subsample, HbA1c >= 6.5 or FPG >= 126).
# Example format: NCHS_P = {"NCHS-style: Total (HbA1c or FPG)": (pct, ci_low, ci_high)}
NCHS_P = {}
for measure, (pub, lo, hi) in NCHS_P.items():
    r = prev[(prev["cycle"] == "2017-2020") & (prev["measure"] == measure)]
    ours = r["pct"].iloc[0]
    val.append({"cycle": "2017-2020", "measure": measure, "ours_pct": round(ours, 1),
                "nchs_pct": pub, "nchs_ci": f"{lo}-{hi}", "match": "PASS" if lo <= ours <= hi else "FAIL"})
if not NCHS_P:
    print("NOTE: 2017-2020 has no published comparison yet — add one to NCHS_P.")

val = pd.DataFrame(val)
val.to_csv(os.path.join(OUT_DIR, "step6_reference_checks.csv"), index=False)
print(val.to_string(index=False))
if len(val) and (val["match"] == "PASS").all():
    print("\nAVAILABLE REFERENCE CHECKS PASSED (2021-2023 and pooled 2013-2016, NCHS-style definition).")
else:
    print("\nREFERENCE CHECK FAILED — fix this before anyone uses the data downstream.")
print("What these checks do NOT cover:\n"
      "  - 2017-March 2020 (no matching published estimate added yet)\n"
      "  - 2011-2012 on its own\n"
      "  - the HbA1c-only labels (y_bio, undx), the screening estimate and the subgroups\n"
      "  - the ML-stage weights. They show the merge, weights and design reproduce NCHS for the\n"
      "    periods and definition checked; they do not prove every downstream choice is correct.")

NOTE: 2017-2020 has no published comparison yet — add one to NCHS_P.
    cycle                          measure  ours_pct  nchs_pct        nchs_ci match
2021-2023 NCHS-style: Total (HbA1c or FPG)      15.8      15.8      13.6-18.2  PASS
2021-2023            NCHS-style: Diagnosed      11.3      11.3       9.3-13.5  PASS
2021-2023          NCHS-style: Undiagnosed       4.5       4.5        3.3-5.9  PASS
2013-2016            2013-16 pooled: Total      14.9      15.0 ours 13.4-16.4  PASS
2013-2016        2013-16 pooled: Diagnosed      10.5      10.5  ours 9.3-11.8  PASS
2013-2016      2013-16 pooled: Undiagnosed       4.4       4.5   ours 3.5-5.2  PASS

AVAILABLE REFERENCE CHECKS PASSED (2021-2023 and pooled 2013-2016, NCHS-style definition).
What these checks do NOT cover:
  - 2017-March 2020 (no matching published estimate added yet)
  - 2011-2012 on its own
  - the HbA1c-only labels (y_bio, undx), the screening estimate and the subgroups
  - the ML-stage weights. They show the merge, we

## 7. LAYER 4 results: case counts and reporting plan

In [8]:
MIN_EVENTS = 30
count_rows = []
adults = nhanes_all[nhanes_all["adult"] & nhanes_all["undx"].notna()]
for cycle, d in adults.groupby("cycle"):
    for var in ["income_band", "income_band2", "insured", "routine_care", "age_band", "age_band2", "sex", "race_eth", "race_eth4"]:
        for level, g in d.groupby(var, observed=True, dropna=False):
            count_rows.append({"cycle": cycle, "period": CYCLES[cycle][4], "subgroup": var, "level": level,
                               "n_adults": len(g), "n_undiagnosed": int(g["undx"].sum()),
                               "n_screened_answered": int(g["screened"].notna().sum())})
counts = pd.DataFrame(count_rows)
counts["too_small"] = counts["n_undiagnosed"] < MIN_EVENTS
counts.to_csv(os.path.join(OUT_DIR, "step7_case_counts.csv"), index=False)
print(counts.pivot_table(index=["subgroup", "level"], columns="cycle",
                         values="n_undiagnosed", aggfunc="sum").to_string())
print(f"\n{int(counts['too_small'].sum())} cycle x subgroup cells have < {MIN_EVENTS} undiagnosed cases.")

# The analysis compares PERIODS (train = 3 pooled cycles, test_P, test_L), so the count that matters
# for power is per period, not per cycle.
by_period = counts.pivot_table(index=["subgroup", "level"], columns="period",
                               values="n_undiagnosed", aggfunc="sum")[["train", "test_P", "test_L"]]
by_period.to_csv(os.path.join(OUT_DIR, "step7_case_counts_by_period.csv"))
print("\nUndiagnosed cases per PERIOD (cells < 30 marked *):")
print(by_period.apply(lambda col: col.map(lambda v: f"{int(v)}*" if v < MIN_EVENTS else str(int(v)))).to_string())


# --- Reporting plan, fixed BEFORE looking at results (edit only by team agreement, and log the change).
# Categories are NOT merged just to reach 30; coarser versions are pre-defined alternatives.
PRIMARY_SUBGROUPS = ["income_band2", "insured", "age_band2", "sex"]   # main comparisons
long = by_period.reset_index().melt(id_vars=["subgroup", "level"], var_name="period", value_name="n_undiagnosed")
long["role"] = np.where(long["subgroup"].isin(PRIMARY_SUBGROUPS), "primary", "exploratory")
long["status"] = np.select(
    [long["role"].eq("primary") & (long["n_undiagnosed"] >= MIN_EVENTS),
     long["role"].eq("primary")],
    ["report", "report with CI, label unstable, no strong claim"],
    default="exploratory only")
long.to_csv(os.path.join(OUT_DIR, "step7_reporting_plan.csv"), index=False)
print("\nReporting plan (pre-specified primary subgroups):")
print(long[long["role"] == "primary"].pivot_table(index=["subgroup", "level"], columns="period",
      values="status", aggfunc="first")[["train", "test_P", "test_L"]].to_string())

cycle                          2011-2012  2013-2014  2015-2016  2017-2020  2021-2023
subgroup     level                                                                  
age_band     20-39                    26         13         16         30         14
             40-59                    72         56         62        118         45
             60+                      71         79         73        136         93
age_band2    20-59                    98         69         78        148         59
             60+                      71         79         73        136         93
income_band  1.3-3.5                  51         52         46         99         74
             <1.3                     66         51         60         66         29
             >3.5                     35         37         23         80         31
income_band2 <1.3                     66         51         60         66         29
             >=1.3                    86         89         69   

## 8. SCHEMA CONTRACT — the column names everyone uses

In [9]:
# If this cell fails, someone (or some LLM) renamed or recoded a column. Do NOT edit SCHEMA to make
# it pass — fix the code instead. SCHEMA changes only by team agreement.
NUM01 = {0.0, 1.0}
SCHEMA = {
    # identifiers and design
    "SEQN": "int", "cycle": set(CYCLES), "period": {"train", "test_P", "test_L"},
    "stratum": "str", "psu": "str",
    # weights (per cycle) and pooled weights
    "w_int": "weight", "w_mec": "weight", "w_lab": "weight", "w_fast": "weight",
    "w": "weight", "w_int_pooled": "weight", "w_fast_pooled": "weight", "pool_factor": {1.0, 1 / 3},
    # population flags
    "adult": "bool", "lab_sample": "bool",
    # labels (1.0 / 0.0 / NaN)
    "y_diag": NUM01, "y_bio": NUM01, "y_bio_any": NUM01, "undx": NUM01,
    "y_bio_fpg": NUM01, "undx_fpg": NUM01, "y_diag_fpg": NUM01, "screened": NUM01,
    # subgroups
    "insured": NUM01, "usual_source_care": NUM01, "routine_care": NUM01,
    # standardised measurements
    "age": "num", "pir": "num", "education": "num", "bmi": "num", "waist": "num",
    "sbp": "num", "dbp": "num", "hba1c": "num", "fpg": "num",
    "bp_method": {"auscultatory", "oscillometric"},
    "income_band": {"<1.3", "1.3-3.5", ">3.5"}, "income_band2": {"<1.3", ">=1.3"},
    "age_band": {"20-39", "40-59", "60+"}, "age_band2": {"20-59", "60+"},
    "sex": {"male", "female"},
    "race_eth": {"Mexican American", "Other Hispanic", "NH White", "NH Black", "NH Asian", "Other/Multi"},
    "race_eth4": {"Hispanic", "NH White", "NH Black", "NH Asian/Other"},
}
problems = []
for col, rule in SCHEMA.items():
    if col not in nhanes_all.columns:
        problems.append(f"missing column: {col}")
        continue
    s = nhanes_all[col]
    if rule == "int":
        ok = pd.api.types.is_integer_dtype(s)
    elif rule == "str":
        ok = s.dropna().map(type).eq(str).all()
    elif rule == "bool":
        ok = pd.api.types.is_bool_dtype(s)
    elif rule == "num":
        ok = pd.api.types.is_numeric_dtype(s)
    elif rule == "weight":
        ok = pd.api.types.is_numeric_dtype(s) and (s.dropna() >= 0).all()
    else:
        vals = set(s.dropna().astype(object).unique())
        vals = {float(v) if isinstance(v, (int, float, np.floating, np.integer)) else str(v) for v in vals}
        allowed = {float(v) if isinstance(v, (int, float)) else str(v) for v in rule}
        ok = vals <= allowed or (rule == {1.0, 1 / 3} and all(min(abs(v - a) for a in allowed) < 1e-9 for v in vals))
        if not ok:
            problems.append(f"{col}: unexpected values {sorted(map(str, vals - allowed))[:5]}")
            continue
    if not ok:
        problems.append(f"{col}: wrong type (expected {rule})")
# The analysis dataset must contain NO raw NHANES names (layer 1 stays in nhanes_raw_merged)
RAW_PATTERN = r"^(SDMV|WT|LBX|LBD|DIQ|DID|RID|RIA|BMX|BPX|HIQ|HUQ|IND|DMD|PHA|SDD)"
raw_left = [c for c in nhanes_all.columns if re.match(RAW_PATTERN, c)]
if raw_left:
    problems.append(f"raw NHANES names in the analysis dataset: {raw_left}")
extra = sorted(set(nhanes_all.columns) - set(SCHEMA))
if extra:
    problems.append(f"columns not in the agreed schema (add by team agreement): {extra}")
if problems:
    raise AssertionError("SCHEMA CONTRACT BROKEN:\n  " + "\n  ".join(problems))
pd.DataFrame([{"column": k, "rule": (sorted(map(str, v)) if isinstance(v, set) else v)}
              for k, v in SCHEMA.items()]).to_csv(os.path.join(OUT_DIR, "schema.csv"), index=False)
print(f"Schema contract OK: all {len(SCHEMA)} shared columns present with agreed names and values.")

Schema contract OK: all 43 shared columns present with agreed names and values.


## 9. IF ANYTHING FAILED: copy this output + the full error and send it

In [10]:
import sys, platform
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__,
      "| platform", platform.platform())
print("rows:", len(df), "| analysis rows:", len(nhanes_all) if "nhanes_all" in globals() else "not built")
print("output files:", sorted(os.listdir(OUT_DIR)))

python 3.14.3 | pandas 3.0.5 | numpy 2.5.2 | platform Windows-11-10.0.26200-SP0
rows: 57395 | analysis rows: 57395
output files: ['nhanes_all.csv', 'nhanes_raw_merged.csv', 'schema.csv', 'step1_download_audit.csv', 'step2_merge_overlap_audit.csv', 'step3_weight_audit.csv', 'step4_missing_hba1c.csv', 'step5_weighted_prevalence.csv', 'step6_reference_checks.csv', 'step7_case_counts.csv', 'step7_case_counts_by_period.csv', 'step7_reporting_plan.csv']


## Still open before research use
1. A published 2017–2020 estimate with the same definition (fill `NCHS_P` in cell 6).
2. HbA1c method notes for GHB_G/H/I/P (2011–12 Tosoh G7 → G8).
3. Cross-check the custom Taylor SEs against a survey package (e.g. R `survey`) for one estimate.
4. Inspect every CSV in `nhanes_out/` from a clean run.